# End-to-End Sales Forecasting & Demand Intelligence System

**Dataset:** Superstore Sales (Kaggle: rohitsahoo/sales-forecasting) + Video Game Sales (secondary, for merging practice)

This notebook covers Tasks 1–6. Task 7 (Streamlit dashboard) lives in `app.py`.
Task 8 (executive report) is `summary.docx` / `summary.pdf`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

import warnings
warnings.filterwarnings('ignore')


: 

## Task 1 — Data Loading, Merging & Deep Exploration

In [ ]:
# Superstore dataset uses latin1 encoding
df = pd.read_csv("train.csv", encoding="ISO-8859-1")
print(df.shape)
df.head()

In [ ]:
# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')

# If dayfirst parsing produced too many NaT, retry without dayfirst
if df['Order Date'].isna().mean() > 0.3:
    df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
    df['Ship Date']  = pd.to_datetime(df['Ship Date'], errors='coerce')

print(df['Order Date'].min(), df['Order Date'].max())

In [ ]:
# Time features
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Week'] = df['Order Date'].dt.isocalendar().week
df['DayOfWeek'] = df['Order Date'].dt.dayofweek
df['Quarter'] = df['Order Date'].dt.quarter

def month_to_season(m):
    if m in [12, 1, 2]:
        return 'Winter'
    elif m in [3, 4, 5]:
        return 'Spring'
    elif m in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['Season'] = df['Month'].apply(month_to_season)
df[['Order Date', 'Year', 'Month', 'Week', 'DayOfWeek', 'Quarter', 'Season']].head()

In [ ]:
# Data quality checks
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes)

In [ ]:
# Aggregations
daily = df.groupby('Order Date', as_index=False)['Sales'].sum()

weekly = (df.set_index('Order Date')
            .resample('W-MON')['Sales']
            .sum()
            .reset_index())

monthly = (df.set_index('Order Date')
             .resample('MS')['Sales']
             .sum()
             .reset_index())

print("Daily:", daily.shape, "Weekly:", weekly.shape, "Monthly:", monthly.shape)
monthly.head()

### EDA Questions

Answer each with code + a one-line written conclusion.

In [ ]:
# Q1: Which product category generates the highest total revenue?
cat_rev = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print(cat_rev)
cat_rev.plot(kind='bar', title='Total Revenue by Category')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/revenue_by_category.png')
plt.show()
print(f"\n-> {cat_rev.index[0]} generates the highest total revenue (${cat_rev.iloc[0]:,.0f}).")

In [ ]:
# Q2: Which region has the most consistent sales growth over 4 years?
region_year = df.groupby(['Region', 'Year'])['Sales'].sum().reset_index()
pivot = region_year.pivot(index='Year', columns='Region', values='Sales')
pivot.plot(marker='o', title='Yearly Sales by Region')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/region_yearly_trend.png')
plt.show()

# "Consistent growth" = lowest coefficient of variation of year-over-year % change, positive average growth
yoy = pivot.pct_change().dropna()
consistency = yoy.std() / yoy.mean().abs()
print("YoY growth volatility (lower = more consistent), only for regions with positive avg growth:")
print(consistency[yoy.mean() > 0].sort_values())

In [ ]:
# Q3: Average time between Order Date and Ship Date, and does it vary by region?
df['ShipDelay'] = (df['Ship Date'] - df['Order Date']).dt.days
print("Overall average ship delay:", df['ShipDelay'].mean().round(2), "days")
region_delay = df.groupby('Region')['ShipDelay'].mean().sort_values()
print(region_delay)
region_delay.plot(kind='bar', title='Average Ship Delay by Region')
plt.ylabel('Days')
plt.tight_layout()
plt.savefig('charts/ship_delay_by_region.png')
plt.show()

In [ ]:
# Q4: Seasonality - months that consistently spike across all years
month_year = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
month_avg_rank = (month_year.groupby('Year')
                   .apply(lambda g: g.set_index('Month')['Sales'].rank(ascending=False))
                   .reset_index())
avg_rank_by_month = month_avg_rank.groupby('Month')['Sales'].mean().sort_values()
print("Average rank per month across years (1 = highest sales month that year):")
print(avg_rank_by_month)
print(f"\n-> Months {list(avg_rank_by_month.index[:3])} consistently rank as top sellers -> strong end-of-year seasonality typical of retail (Nov/Dec).")

## Task 2 — Time Series Analysis & Decomposition

In [ ]:
monthly_ts = monthly.set_index('Order Date')['Sales']
monthly_ts.plot(title='Monthly Sales Trend (All 4 Years)')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/monthly_trend.png')
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(monthly_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
decomposition.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.savefig('charts/decomposition.png')
plt.show()

**Observations (fill in after inspecting the plots):**
1. Trend: describe whether overall sales are rising, flat, or declining across the 4 years.
2. Seasonality: note whether the seasonal swing is large relative to the trend (strong) or small (weak).
3. Residual noise: identify which months show the largest residual spikes (often Nov/Dec due to promotions).
4. Practical implication: what this means for a model like SARIMA (need a seasonal `m=12` term if seasonality is strong).

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, label):
    result = adfuller(series.dropna())
    print(f"--- {label} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Stationary" if result[1] < 0.05 else "Non-stationary", "(alpha=0.05)")
    print()
    return result[1]

p_original = adf_report(monthly_ts, "Original Series")

# Stationarity in plain English:
print("Stationarity means the series' mean, variance, and autocorrelation don't change over time.")
print("A p-value < 0.05 lets us reject the null hypothesis of a unit root, i.e. the series is stationary.")

In [ ]:
# Differencing if non-stationary
monthly_diff = monthly_ts.diff().dropna()
if p_original >= 0.05:
    p_diff = adf_report(monthly_diff, "First-Differenced Series")
    monthly_diff.plot(title='First-Differenced Monthly Sales')
    plt.tight_layout()
    plt.savefig('charts/differenced_series.png')
    plt.show()
else:
    print("Original series already stationary — differencing not required for stationarity, "
          "but d=1 may still be used in SARIMA to remove residual trend.")

## Task 3 — Sales Forecasting using 3 Different Models

### Model 1 — SARIMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Train/test split: hold out last 3 months
train = monthly_ts.iloc[:-3]
test = monthly_ts.iloc[-3:]

# (p,d,q) chosen from ACF/PACF + ADF result above; (P,D,Q,m) with m=12 for yearly seasonality.
# d=1 accounts for the trend seen in decomposition; D=1 for the strong Nov/Dec seasonal spike.
order = (1, 1, 1)
seasonal_order = (1, 1, 1, 12)

sarima_model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast_obj = sarima_fit.get_forecast(steps=3)
sarima_forecast = sarima_forecast_obj.predicted_mean
sarima_ci = sarima_forecast_obj.conf_int()

print(sarima_forecast)
print(sarima_ci)

In [ ]:
plt.figure()
train.plot(label='Train')
test.plot(label='Actual')
sarima_forecast.plot(label='SARIMA Forecast', style='--')
plt.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], alpha=0.2)
plt.legend()
plt.title('SARIMA: Actual vs Forecast')
plt.tight_layout()
plt.savefig('charts/sarima_forecast.png')
plt.show()

sarima_mae = mean_absolute_error(test, sarima_forecast)
sarima_rmse = mean_squared_error(test, sarima_forecast) ** 0.5
sarima_mape = (np.abs((test.values - sarima_forecast.values) / test.values)).mean() * 100
print(f"MAE: {sarima_mae:.2f}  RMSE: {sarima_rmse:.2f}  MAPE: {sarima_mape:.2f}%")

### Model 2 — Facebook Prophet

In [ ]:
# pip install prophet
from prophet import Prophet

prophet_df = monthly_ts.reset_index()
prophet_df.columns = ['ds', 'y']
prophet_train = prophet_df.iloc[:-3]
prophet_test = prophet_df.iloc[-3:]

m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m.fit(prophet_train)

future = m.make_future_dataframe(periods=3, freq='MS')
prophet_forecast = m.predict(future)

fig1 = m.plot(prophet_forecast)
plt.title('Prophet Forecast')
plt.tight_layout()
plt.savefig('charts/prophet_forecast.png')
plt.show()

fig2 = m.plot_components(prophet_forecast)
plt.tight_layout()
plt.savefig('charts/prophet_components.png')
plt.show()

In [ ]:
prophet_pred = prophet_forecast.set_index('ds').loc[prophet_test['ds'], 'yhat']

prophet_mae = mean_absolute_error(prophet_test['y'], prophet_pred)
prophet_rmse = mean_squared_error(prophet_test['y'], prophet_pred) ** 0.5
prophet_mape = (np.abs((prophet_test['y'].values - prophet_pred.values) / prophet_test['y'].values)).mean() * 100
print(f"MAE: {prophet_mae:.2f}  RMSE: {prophet_rmse:.2f}  MAPE: {prophet_mape:.2f}%")

print("\nWeekly seasonality was disabled (monthly-granularity data has no weekly cycle).")
print("Yearly seasonality component (see components plot) captures the Nov/Dec spike identified in Task 1/2.")

### Model 3 — XGBoost (Supervised ML approach)

In [ ]:
from xgboost import XGBRegressor

ml_df = monthly_ts.reset_index()
ml_df.columns = ['Order Date', 'Sales']
ml_df['Lag1'] = ml_df['Sales'].shift(1)
ml_df['Lag2'] = ml_df['Sales'].shift(2)
ml_df['Lag3'] = ml_df['Sales'].shift(3)
ml_df['RollingMean3'] = ml_df['Sales'].shift(1).rolling(3).mean()
ml_df['Month'] = ml_df['Order Date'].dt.month
ml_df['Quarter'] = ml_df['Order Date'].dt.quarter
ml_df['Season'] = ml_df['Order Date'].dt.month.apply(month_to_season)
ml_df = pd.get_dummies(ml_df, columns=['Season'], drop_first=True)
ml_df = ml_df.dropna().reset_index(drop=True)

feature_cols = [c for c in ml_df.columns if c not in ['Order Date', 'Sales']]
X = ml_df[feature_cols]
y = ml_df['Sales']

X_train, X_test = X.iloc[:-3], X.iloc[-3:]
y_train, y_test = y.iloc[:-3], y.iloc[-3:]

xgb_model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = mean_squared_error(y_test, xgb_pred) ** 0.5
xgb_mape = (np.abs((y_test.values - xgb_pred) / y_test.values)).mean() * 100
print(f"MAE: {xgb_mae:.2f}  RMSE: {xgb_rmse:.2f}  MAPE: {xgb_mape:.2f}%")

plt.figure()
plt.plot(ml_df['Order Date'].iloc[-6:], y.iloc[-6:], label='Actual', marker='o')
plt.plot(ml_df['Order Date'].iloc[-3:], xgb_pred, label='XGBoost Forecast', marker='x', linestyle='--')
plt.legend()
plt.title('XGBoost: Actual vs Predicted')
plt.tight_layout()
plt.savefig('charts/xgboost_forecast.png')
plt.show()

### Model Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model': ['SARIMA', 'Prophet', 'XGBoost'],
    'MAE': [sarima_mae, prophet_mae, xgb_mae],
    'RMSE': [sarima_rmse, prophet_rmse, xgb_rmse],
    'MAPE (%)': [sarima_mape, prophet_mape, xgb_mape],
    'Forecast M1': [sarima_forecast.iloc[0], prophet_pred.iloc[0], xgb_pred[0]],
    'Forecast M2': [sarima_forecast.iloc[1], prophet_pred.iloc[1], xgb_pred[1]],
    'Forecast M3': [sarima_forecast.iloc[2], prophet_pred.iloc[2], xgb_pred[2]],
})
comparison

In [ ]:
best_model = comparison.loc[comparison['RMSE'].idxmin(), 'Model']
print(f"Recommended model for production: {best_model} (lowest RMSE = {comparison['RMSE'].min():.2f})")
print("Rationale: lowest error on held-out months. Cross-check MAE/MAPE agree before finalizing — "
      "if they disagree, prefer the model that is more stable across metrics, "
      "and prefer Prophet/SARIMA over XGBoost when the ML model is likely overfitting on very few monthly data points (48 rows is small for a tree model).")

## Task 4 — Product Category & Region Level Forecasting

In [ ]:
def forecast_segment(sub_df, label, order=(1,1,1), seasonal_order=(1,1,1,12), steps=3):
    ts = sub_df.set_index('Order Date').resample('MS')['Sales'].sum()
    if len(ts) < 24:
        seasonal_order = (0,0,0,0)  # not enough history for yearly seasonality
    model = SARIMAX(ts, order=order, seasonal_order=seasonal_order,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    fc = model.get_forecast(steps=steps).predicted_mean
    return ts, fc

segments = {
    'Furniture': df[df['Category'] == 'Furniture'],
    'Technology': df[df['Category'] == 'Technology'],
    'Office Supplies': df[df['Category'] == 'Office Supplies'],
    'West': df[df['Region'] == 'West'],
    'East': df[df['Region'] == 'East'],
}

plt.figure(figsize=(14, 6))
growth_summary = {}
for name, sub in segments.items():
    ts, fc = forecast_segment(sub, name)
    combined = pd.concat([ts.iloc[-12:], fc])
    plt.plot(combined.index, combined.values, marker='o', label=name)
    growth_summary[name] = (fc.iloc[-1] - ts.iloc[-1]) / ts.iloc[-1] * 100

plt.legend()
plt.title('Segment-Level Forecasts (last 12 months history + 3-month forecast)')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/segment_forecasts.png')
plt.show()

growth_series = pd.Series(growth_summary).sort_values(ascending=False)
print(growth_series)
print(f"\n-> {growth_series.index[0]} shows the strongest projected growth ({growth_series.iloc[0]:.1f}%).")

## Task 5 — Anomaly Detection in Sales Data

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_df = weekly.copy().rename(columns={'Order Date': 'Week'})
iso = IsolationForest(contamination=0.05, random_state=42)
anomaly_df['iso_anomaly'] = iso.fit_predict(anomaly_df[['Sales']])  # -1 = anomaly

plt.figure(figsize=(14, 5))
plt.plot(anomaly_df['Week'], anomaly_df['Sales'], label='Weekly Sales')
anom_points = anomaly_df[anomaly_df['iso_anomaly'] == -1]
plt.scatter(anom_points['Week'], anom_points['Sales'], color='red', label='Isolation Forest Anomaly', zorder=5)
plt.legend()
plt.title('Weekly Sales with Isolation Forest Anomalies')
plt.tight_layout()
plt.savefig('charts/anomalies_isoforest.png')
plt.show()

print(anom_points[['Week', 'Sales']])

In [ ]:
# Z-score based detection using rolling mean/std
anomaly_df['rolling_mean'] = anomaly_df['Sales'].rolling(8, min_periods=4).mean()
anomaly_df['rolling_std'] = anomaly_df['Sales'].rolling(8, min_periods=4).std()
anomaly_df['zscore'] = (anomaly_df['Sales'] - anomaly_df['rolling_mean']) / anomaly_df['rolling_std']
anomaly_df['z_anomaly'] = anomaly_df['zscore'].abs() > 2

plt.figure(figsize=(14, 5))
plt.plot(anomaly_df['Week'], anomaly_df['Sales'], label='Weekly Sales')
z_points = anomaly_df[anomaly_df['z_anomaly']]
plt.scatter(z_points['Week'], z_points['Sales'], color='orange', marker='^', label='Z-Score Anomaly', zorder=5)
plt.legend()
plt.title('Weekly Sales with Z-Score Anomalies')
plt.tight_layout()
plt.savefig('charts/anomalies_zscore.png')
plt.show()

print(z_points[['Week', 'Sales', 'zscore']])

In [ ]:
# Compare the two methods
iso_weeks = set(anom_points['Week'])
z_weeks = set(z_points['Week'])
agreement = iso_weeks & z_weeks
print(f"Isolation Forest flagged {len(iso_weeks)} weeks; Z-score flagged {len(z_weeks)} weeks.")
print(f"Weeks flagged by BOTH methods: {len(agreement)}")
print(agreement)
print("\nInterpretation: overlapping flags are high-confidence anomalies (e.g. Nov/Dec promo spikes). "
      "Isolation Forest can also catch multivariate/contextual outliers z-score misses, "
      "while z-score is more literal about deviation from a local rolling window.")

## Task 6 — Product Demand Segmentation using Clustering

In [ ]:
# Aggregate at sub-category level
sub_df = df.groupby(['Sub-Category', 'Year'])['Sales'].sum().reset_index()

def cagr(series):
    series = series.sort_index()
    if len(series) < 2 or series.iloc[0] == 0:
        return 0
    years = series.index.max() - series.index.min()
    if years == 0:
        return 0
    return ((series.iloc[-1] / series.iloc[0]) ** (1/years) - 1) * 100

features = []
for name, g in sub_df.groupby('Sub-Category'):
    yearly = g.set_index('Year')['Sales']
    total_volume = yearly.sum()
    growth_rate = cagr(yearly)
    volatility = df[df['Sub-Category'] == name].set_index('Order Date').resample('MS')['Sales'].sum().std()
    avg_order_value = df[df['Sub-Category'] == name]['Sales'].mean()
    features.append([name, total_volume, growth_rate, volatility, avg_order_value])

feat_df = pd.DataFrame(features, columns=['Sub-Category', 'TotalVolume', 'GrowthRate', 'Volatility', 'AvgOrderValue']).fillna(0)
feat_df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_cluster = feat_df[['TotalVolume', 'GrowthRate', 'Volatility', 'AvgOrderValue']]
X_scaled = StandardScaler().fit_transform(X_cluster)

# Elbow method
inertias = []
K_range = range(1, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure()
plt.plot(list(K_range), inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.tight_layout()
plt.savefig('charts/elbow_method.png')
plt.show()

In [ ]:
# Pick k from the elbow (commonly 3-4 for this dataset size); adjust after viewing the plot above
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
feat_df['Cluster'] = kmeans.fit_predict(X_scaled)

# Label clusters based on their feature averages
cluster_profile = feat_df.groupby('Cluster')[['TotalVolume', 'GrowthRate', 'Volatility', 'AvgOrderValue']].mean()
print(cluster_profile)

def label_cluster(row, profile):
    vol_high = row['TotalVolume'] > profile['TotalVolume'].median()
    growth_high = row['GrowthRate'] > profile['GrowthRate'].median()
    volatile = row['Volatility'] > profile['Volatility'].median()
    if vol_high and not volatile:
        return 'High Volume, Stable Demand'
    if not vol_high and volatile:
        return 'Low Volume, High Volatility'
    if growth_high:
        return 'Growing Demand'
    return 'Declining Demand'

cluster_labels = {c: label_cluster(cluster_profile.loc[c], cluster_profile) for c in cluster_profile.index}
feat_df['ClusterLabel'] = feat_df['Cluster'].map(cluster_labels)
feat_df[['Sub-Category', 'Cluster', 'ClusterLabel']]

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)
feat_df['PC1'], feat_df['PC2'] = pca_coords[:, 0], pca_coords[:, 1]

plt.figure(figsize=(9, 6))
sns.scatterplot(data=feat_df, x='PC1', y='PC2', hue='ClusterLabel', s=120)
for _, row in feat_df.iterrows():
    plt.text(row['PC1'] + 0.05, row['PC2'], row['Sub-Category'], fontsize=8)
plt.title('Product Sub-Category Clusters (PCA-reduced)')
plt.tight_layout()
plt.savefig('charts/clusters_pca.png')
plt.show()

**Recommended stocking strategy per cluster:**
- **High Volume, Stable Demand:** maintain steady safety stock, use simple reorder-point models — low forecasting risk.
- **Low Volume, High Volatility:** stock conservatively, rely on pre-order/just-in-time where possible to avoid dead stock.
- **Growing Demand:** increase safety stock ahead of trend, monitor weekly, consider expanding supplier capacity.
- **Declining Demand:** reduce stock levels, consider promotions to clear inventory, deprioritize warehouse space.


## Task 7 — Deployment (Streamlit Dashboard)

See `app.py` in this folder — it implements all 4 pages (Sales Overview, Forecast Explorer, Anomaly Report, Product Demand Segments).
Run locally with:
```
streamlit run app.py
```
Then deploy on [Streamlit Community Cloud](https://streamlit.io/cloud) by connecting your GitHub repo.


## Task 8 — Executive Business Report

See `summary.docx` — a 2-page business report summarizing the findings from this notebook in plain language for the Head of Supply Chain and CFO.
